# Multivariate structured additive model

This example fits several related responses with one additive predictor. A random-walk penalty shares information across response dimensions, while each term keeps its usual covariate-side basis and penalty.

In [ ]:
import jax.numpy as jnp
import liesel.goose as gs
import liesel.model as lsl
import numpy as np
import pandas as pd
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel_gam as gam

We simulate four related outcomes. Their effects sum to zero across dimensions, matching the constraint applied below.

In [ ]:
rng = np.random.default_rng(42)
n = 80
ndim = 4
x = np.linspace(-1.0, 1.0, n)
group = np.resize(np.array(["a", "b", "c", "d"]), n)
profile = np.array([-1.5, -0.5, 0.5, 1.5])
group_shift = np.array([-0.2, 0.1, 0.25, -0.15])
mean = np.sin(np.pi * x)[:, None] * profile
mean += group_shift[pd.Categorical(group).codes, None] * profile
y = mean + rng.normal(scale=0.4, size=mean.shape)
data = pd.DataFrame({"x": x, "group": pd.Categorical(group)})

print(y.shape)
print(data.head(3))

The predictor owns the cross-dimensional penalty and constraint. The linked multivariate term builder reuses both for every term.

In [ ]:
predictor = gam.MVAdditivePredictor.from_random_walk("mu", ndim=ndim)
predictor.constrain("sumzero_coef")

term_builder = gam.TermBuilder.from_df(data)
builder = gam.MVTermBuilder.from_predictor(predictor, term_builder)

smooth = builder.ps(
    "x", k=8, scale=1.0, dimension_scale=1.0
)
group_effect = builder.ri(
    "group", scale=1.0, dimension_scale=1.0
)
predictor += [smooth, group_effect]

print(predictor)
print("dimensions:", predictor.ndim, predictor.latent_ndim)
print("smooth shapes:", smooth.latent.value.shape, smooth.value.shape)

The likelihood sees the reconstructed four-dimensional predictor. Ordinary Liesel inference specifications attached by the builders can then be used without a separate multivariate fitting interface.

In [ ]:
response = lsl.Var.new_obs(
    y,
    distribution=lsl.Dist(
        tfd.MultivariateNormalDiag,
        loc=predictor,
        scale_diag=jnp.full(ndim, 0.4),
    ),
    name="y",
)
model = lsl.Model([response])

print(response.value.shape)
print(sorted(predictor.terms))

In [ ]:
engine_builder = gs.LieselMCMC(model).get_engine_builder(
    seed=13, num_chains=2
)
engine_builder.add_burnin(200)
engine_builder.add_posterior(400)
engine = engine_builder.build()
engine.sample_all_epochs()
results = engine.get_results()
samples = results.get_posterior_samples()

print(samples[smooth.coef.name].shape)

For prediction, categorical labels are encoded with the mapping established from the training registry.

In [ ]:
newdata = builder.labels_to_integers(
    {
        "x": np.array([-0.75, 0.0, 0.75]),
        "group": np.array(["a", "b", "d"]),
    }
)
prediction = predictor.predict(
    gs.Position(samples), newdata=gs.Position(newdata)
)

print(prediction.shape)